In [ ]:
import torch
import torch.nn as nn
import os

print("=" * 50)
print("ENVIRONMENT INFO")
print("=" * 50)

print("\nPyTorch version:", torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device tersedia:", device)
print("Jumlah GPU:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("✓ GPU aktif! Training akan menggunakan GPU.")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("⚠ GPU tidak terdeteksi")
    print("Jika di RunPod, pastikan pod sudah dijalankan dengan GPU")

# Check CUDA
print("\nCUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)

# Memory info
print("\n" + "=" * 50)

In [ ]:
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Dataset/road-efficientnetb0")
DATASET_PATH = PROJECT_DIR / "data" / "raw" / "road"

print("Dataset folder:", DATASET_PATH)

all_images = []

for split in ["train", "test", "valid"]:
    split_path = DATASET_PATH / split

    if split_path.exists():
        images = [
            str(img_path)
            for img_path in split_path.glob("*")
            if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ]

        all_images.extend(images)
        print(f"{split}: {len(images)} gambar")

    else:
        print(f"{split} tidak ditemukan")

print("-"*30)
print("Total seluruh gambar:", len(all_images))

print("\nContoh gambar:")
for img in all_images[:5]:
    print(img)

In [ ]:
import cv2
import matplotlib.pyplot as plt

# ambil 5 gambar pertama
sample_images = all_images[:5]

print("Total sample:", len(sample_images))

plt.figure(figsize=(15,10))

for i, sample_path in enumerate(sample_images):
    print(f"Gambar {i+1}: {sample_path}")

    # load image
    img = cv2.imread(sample_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(2,3,i+1)
    plt.imshow(img)
    plt.title(f"Original {i+1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15,8))

for i, sample_path in enumerate(sample_images):
    img = cv2.imread(sample_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w, _ = img.shape

    # crop bagian bawah
    roi = img[int(h*0.2):h, :]

    plt.subplot(2,3,i+1)
    plt.imshow(roi)
    plt.title(f"ROI {i+1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15,8))

for i, sample_path in enumerate(sample_images):
    img = cv2.imread(sample_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w, _ = img.shape
    roi = img[int(h*0.4):h, :]

    roi_resized = cv2.resize(roi, (224,224))

    plt.subplot(2,3,i+1)
    plt.imshow(roi_resized)
    plt.title(f"Resize {i+1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15,8))

for i, sample_path in enumerate(sample_images):
    img = cv2.imread(sample_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w, _ = img.shape
    roi = img[int(h*0.4):h, :]
    roi = cv2.resize(roi, (224,224))

    gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)

    plt.subplot(2,3,i+1)
    plt.imshow(blur, cmap="gray")
    plt.title(f"Blur {i+1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15,8))

for i, sample_path in enumerate(sample_images):
    img = cv2.imread(sample_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w, _ = img.shape
    roi = img[int(h*0.4):h, :]
    roi = cv2.resize(roi, (224,224))

    gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)

    edges = cv2.Canny(blur, 50, 150)

    plt.subplot(2,3,i+1)
    plt.imshow(edges, cmap="gray")
    plt.title(f"Segmentasi {i+1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

feature_extractor = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    pooling='avg'
)

print("Feature extractor berhasil dimuat")

import numpy as np

sample_path = sample_images[0]

img = cv2.imread(sample_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

h, w, _ = img.shape
roi = img[int(h*0.4):h, :]
roi = cv2.resize(roi, (224,224))

gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)
blur = cv2.GaussianBlur(gray, (5,5), 0)

edges = cv2.Canny(blur, 50, 150)

segmented = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)

img_array = np.expand_dims(segmented, axis=0)
img_array = preprocess_input(img_array)

feature = feature_extractor.predict(img_array)

print("Shape feature:")
print(feature.shape)
print(feature.flatten()[:20])

In [ ]:
features = []
valid_images = []

for img_path in all_images:
    try:
        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        h, w, _ = img.shape

        # ROI
        roi = img[int(h*0.4):h, :]
        roi = cv2.resize(roi, (224,224))

        # Segmentasi
        gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)
        blur = cv2.GaussianBlur(gray, (5,5), 0)
        edges = cv2.Canny(blur, 50, 150)

        segmented = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)

        # preprocessing EfficientNet
        img_array = np.expand_dims(segmented, axis=0)
        img_array = preprocess_input(img_array)

        # feature extraction
        feature = feature_extractor.predict(
            img_array,
            verbose=0
        )

        features.append(feature.flatten())
        valid_images.append(img_path)

    except Exception as e:
        print(f"Error pada {img_path}: {e}")

In [ ]:
features = np.array(features)

print("Shape seluruh feature vector:")
print(features.shape)

In [ ]:
import pandas as pd

df_features = pd.DataFrame(features)

df_features["image_path"] = valid_images

df_features.to_csv(
    "/content/drive/MyDrive/Dataset/road-efficientnetb0/features.csv",
    index=False
)

print("Feature vector berhasil disimpan")